<a href="https://colab.research.google.com/github/zxn-999/BSE_CNN/blob/main/Unet_3_22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install segmentation-models-pytorch --quiet
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from sklearn.metrics import confusion_matrix
from matplotlib.colors import ListedColormap

# 自定义颜色（4个phase）
colors = [
    (0, 0, 0),        # Class 0 - 黑色 (孔隙)
    (1, 0, 0),        # Class 1 - 红色
    (0, 1, 0),        # Class 2 - 绿色
    (1, 1, 0)         # Class 3 - 蓝色
]

cmap = ListedColormap(colors)

def save_tif_gray(path, img):
    if img.dtype != np.uint8:
        img = (img * 255).astype(np.uint8)
    cv2.imwrite(path, img)

def save_tif_color(path, label_map, cmap):
    colored = cmap(label_map)[:, :, :3]
    colored = (colored * 255).astype(np.uint8)
    colored = cv2.cvtColor(colored, cv2.COLOR_RGB2BGR)
    cv2.imwrite(path, colored)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.8 MB/s eta 0:00:00


In [3]:
# 1. 基础配置与路径
# ==========================================
IMAGE_DIR = "/content/drive/MyDrive/BSE/images"
MASK_DIR = "/content/drive/MyDrive/BSE/mask"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
EPOCHS = 10
PATCH_SIZE = 256
NUM_CLASSES = 4

# 创建保存结果的文件夹
OUTPUT_DIR = "/content/drive/MyDrive/BSE/output_Unet"

VAL_IMG_DIR = os.path.join(OUTPUT_DIR, "validate/images")
VAL_GT_DIR  = os.path.join(OUTPUT_DIR, "validate/gt")
VAL_PRED_DIR= os.path.join(OUTPUT_DIR, "validate/pred")

TEST_IMG_DIR = os.path.join(OUTPUT_DIR, "test/images")
TEST_GT_DIR  = os.path.join(OUTPUT_DIR, "test/gt")
TEST_PRED_DIR= os.path.join(OUTPUT_DIR, "test/pred")

In [4]:
# 2. 数据集类定义
# ==========================================
class BSEPatchDataset(Dataset):
    def __init__(self, image_dir, mask_dir, image_list, patch_size=256):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.patch_size = patch_size
        self.image_files = image_list
        self.patches = []
        self._prepare_patches()

    def _prepare_patches(self):
        for img_name in self.image_files:
            img_path = os.path.join(self.image_dir, img_name)
            mask_path = os.path.join(self.mask_dir, img_name.replace(".tif", "_mask.tif"))

            if not os.path.exists(mask_path):
                continue

            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            h, w = image.shape

            for y in range(0, h - self.patch_size + 1, self.patch_size):
                for x in range(0, w - self.patch_size + 1, self.patch_size):
                    self.patches.append((img_path, mask_path, x, y))

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        img_path, mask_path, x, y = self.patches[idx]

        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        image = image[y:y+self.patch_size, x:x+self.patch_size]
        mask = mask[y:y+self.patch_size, x:x+self.patch_size]

        image = torch.from_numpy(image).float().unsqueeze(0) / 255.0
        mask = torch.from_numpy(mask).long()

        mask = torch.clamp(mask, min=0, max=3)

        return image, mask

In [5]:
# 3. 评价指标计算函数
# ==========================================
def calculate_metrics(pred, target, num_classes=4):
    """计算准确率、精确率、召回率、假阳性率"""
    pred = pred.view(-1).cpu().numpy()
    target = target.view(-1).cpu().numpy()
    cm = confusion_matrix(target, pred, labels=range(num_classes))

    results = []
    for i in range(num_classes):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - tp - fn - fp

        accuracy = (tp + tn) / np.sum(cm) if np.sum(cm) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        fall_out = fp / (fp + tn) if (fp + tn) > 0 else 0
    # 增加IoU和F1score
        IoU = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
        F1 = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
        results.append([accuracy, precision, recall, fall_out, IoU, F1])
    return np.array(results)

In [8]:
# 4. 数据划分与加载
# ==========================================
all_images = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith(".tif")])

# 固定随机种子（保证可复现）
np.random.seed(42)
np.random.shuffle(all_images)

n = len(all_images)

train_imgs = all_images[:int(0.7 * n)]
val_imgs   = all_images[int(0.7 * n):int(0.85 * n)]
test_imgs  = all_images[int(0.85 * n):]

print("Train images:", len(train_imgs))
print("Val images:", len(val_imgs))
print("Test images:", len(test_imgs))

# 创建 dataset
train_ds = BSEPatchDataset(IMAGE_DIR, MASK_DIR, train_imgs, PATCH_SIZE)
val_ds   = BSEPatchDataset(IMAGE_DIR, MASK_DIR, val_imgs, PATCH_SIZE)
test_ds  = BSEPatchDataset(IMAGE_DIR, MASK_DIR, test_imgs, PATCH_SIZE)


train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

print("Train patches:", len(train_ds))
print("Val patches:", len(val_ds))
print("Test patches:", len(test_ds))

Train images: 51
Val images: 11
Test images: 12
Train patches: 6455
Val patches: 1096
Test patches: 1087


In [9]:
# 4.5 计算类别权重（基于数据分布）
# ==========================================
print("Calculating class weights...")

class_counts = np.zeros(NUM_CLASSES)

for i in range(len(train_ds)):
    _, mask = train_ds[i]
    m = mask.numpy()
    for c in range(NUM_CLASSES):
        class_counts[c] += np.sum(m == c)

class_freq = class_counts / np.sum(class_counts)

# 使用 sqrt 平滑（推荐）
weights = 1 / np.sqrt(class_freq + 1e-6)
weights = weights / weights.min()

print("Class frequencies:", class_freq)
print("Class weights:", weights)

weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

Calculating class weights...
Class frequencies: [0.10960968 0.34146629 0.42896865 0.11995538]
Class weights: [1.97827586 1.12082737 1.         1.89104388]


In [10]:
# 5. 模型、损失函数与优化器
# ==========================================
model = smp.Unet(encoder_name="resnet34", in_channels=1, classes=NUM_CLASSES).to(DEVICE)
dice_loss = smp.losses.DiceLoss(mode='multiclass')
ce_loss = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [11]:
# 6. 训练与验证循环
# ==========================================
for epoch in range(EPOCHS):
    # --- 训练阶段 ---
    model.train()
    train_loss = 0
    running_loss_d = 0
    running_loss_ce = 0
    for images, masks in train_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss_d = dice_loss(outputs, masks)
        loss_ce = ce_loss(outputs, masks)
        loss = loss_d + loss_ce
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        running_loss_d += loss_d.item() * images.size(0)
        running_loss_ce += loss_ce.item() * images.size(0)

    # --- 验证阶段 ---
    model.eval()
    val_metrics = np.zeros((NUM_CLASSES, 6))
    with torch.no_grad():
        for i, (images, masks) in enumerate(val_loader):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            val_metrics += calculate_metrics(preds, masks)
            # 每个 Epoch 保存前 3 个验证结果图
            if i < 3:
                img_np = images[0,0].cpu().numpy()
                pred_np = preds[0].cpu().numpy()
                mask_np = masks[0].cpu().numpy()

                save_tif_gray(
                    os.path.join(VAL_IMG_DIR, f"epoch{epoch}_sample{i}.tif"),
                    img_np
                )

                save_tif_color(
                    os.path.join(VAL_GT_DIR, f"epoch{epoch}_sample{i}.tif"),
                    mask_np, cmap
                )

                save_tif_color(
                    os.path.join(VAL_PRED_DIR, f"epoch{epoch}_sample{i}.tif"),
                    pred_np, cmap
                )
    avg_val_metrics = val_metrics / len(val_loader)
    epoch_loss_d = running_loss_d / len(train_loader)
    epoch_loss_ce = running_loss_ce / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] "
      f"Total: {train_loss/len(train_loader):.4f} | "
      f"Dice: {epoch_loss_d:.4f} | "
      f"CE: {epoch_loss_ce:.4f}")
    print(f"\n--- Epoch [{epoch+1}/{EPOCHS}] Validation Summary ---")
    for c in range(NUM_CLASSES):
        print(f"Class {c}: Recall={avg_val_metrics[c, 2]:.4f}, Fall-out={avg_val_metrics[c, 3]:.4f}, IoU={avg_val_metrics[c,4]:.4f}, F1={avg_val_metrics[c,5]:.4f}")
    print("-" * 30)

Epoch [1/10] Total: 6.8465 | Dice: 2.6306 | CE: 4.2159

--- Epoch [1/10] Validation Summary ---
Class 0: Recall=0.6691, Fall-out=0.0035, IoU=0.6342, F1=0.7486
Class 1: Recall=0.7788, Fall-out=0.0619, IoU=0.6988, F1=0.8147
Class 2: Recall=0.9024, Fall-out=0.1570, IoU=0.7595, F1=0.8596
Class 3: Recall=0.9266, Fall-out=0.0367, IoU=0.6822, F1=0.7952
------------------------------
Epoch [2/10] Total: 4.5051 | Dice: 1.6880 | CE: 2.8172

--- Epoch [2/10] Validation Summary ---
Class 0: Recall=0.9276, Fall-out=0.0151, IoU=0.7726, F1=0.8588
Class 1: Recall=0.9259, Fall-out=0.0808, IoU=0.8020, F1=0.8866
Class 2: Recall=0.8622, Fall-out=0.0489, IoU=0.8112, F1=0.8931
Class 3: Recall=0.8500, Fall-out=0.0176, IoU=0.7357, F1=0.8375
------------------------------
Epoch [3/10] Total: 3.9600 | Dice: 1.4740 | CE: 2.4860

--- Epoch [3/10] Validation Summary ---
Class 0: Recall=0.9729, Fall-out=0.0242, IoU=0.7526, F1=0.8463
Class 1: Recall=0.8886, Fall-out=0.0588, IoU=0.7938, F1=0.8825
Class 2: Recall=0.85

In [12]:
# 7. 最终测试阶段
# ==========================================
print("\n--- Starting Final Test ---")
import cv2
import os
model.eval()
# 用于累加所有 Patch 的指标
test_metrics_accumulator = np.zeros((NUM_CLASSES, 6)) # [Accuracy, Precision, Recall, Fall-out, IoU, F1]
with torch.no_grad():
    for i, (images, masks) in enumerate(test_loader):
        img_np = images[0,0].cpu().numpy()
        pred_np = preds[0].cpu().numpy()
        mask_np = masks[0].cpu().numpy()

        # ===== 原图 =====
        save_tif_gray(
            os.path.join(TEST_IMG_DIR, f"test_{i}.tif"),
            img_np
        )

        # ===== GT =====
        save_tif_color(
            os.path.join(TEST_GT_DIR, f"test_{i}.tif"),
            mask_np, cmap
        )

        # ===== Prediction =====
        save_tif_color(
            os.path.join(TEST_PRED_DIR, f"test_{i}.tif"),
            pred_np, cmap
        )

        # 计算当前 Patch 的指标并累加
        patch_metrics = calculate_metrics(preds, masks, num_classes=NUM_CLASSES)
        test_metrics_accumulator += patch_metrics

        # 保存所有测试集的预测结果图
        plt.savefig(os.path.join(OUTPUT_DIR, "test", f"test_{i}_color.tif"),
            bbox_inches='tight', pad_inches=0)

# 计算所有测试样本的平均值
final_metrics = test_metrics_accumulator / len(test_loader)
# --- 打印每个 Class 的单独指标 ---
print(f"{'Class':<10} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10} | {'Fall-out':<10} | {'IoU':<10} | {'F1':<10}")
print("-" * 65)
for c in range(NUM_CLASSES):
    print(f"Class {c:<6} | {final_metrics[c,0]:.4f}   | {final_metrics[c,1]:.4f}    | {final_metrics[c,2]:.4f} | {final_metrics[c,3]:.4f} | {final_metrics[c,4]:.4f} | {final_metrics[c,5]:.4f}")

# --- 计算并打印总指标 (Macro Average) ---
# 总指标即所有类别指标的平均值
total_accuracy  = np.mean(final_metrics[:, 0])
total_precision = np.mean(final_metrics[:, 1])
total_recall    = np.mean(final_metrics[:, 2])
total_fall_out  = np.mean(final_metrics[:, 3])
total_IoU       = np.mean(final_metrics[:, 4])
total_F1        = np.mean(final_metrics[:, 5])

print("-" * 65)
print(f"{'Average':<10} | {total_accuracy:.4f}   | {total_precision:.4f}    | {total_recall:.4f} | {total_fall_out:.4f}  | {total_IoU:.4f}  | {total_F1:.4f}")
print("="*50)


--- Starting Final Test ---
Class      | Accuracy   | Precision  | Recall     | Fall-out   | IoU        | F1        
-----------------------------------------------------------------
Class 0      | 0.8275   | 0.1267    | 0.0668 | 0.0639 | 0.0390 | 0.0740
Class 1      | 0.5540   | 0.3519    | 0.3095 | 0.3125 | 0.1910 | 0.3173
Class 2      | 0.4976   | 0.4148    | 0.4947 | 0.5139 | 0.2896 | 0.4410
Class 3      | 0.8063   | 0.1039    | 0.1078 | 0.1102 | 0.0456 | 0.0837
-----------------------------------------------------------------
Average    | 0.6714   | 0.2493    | 0.2447 | 0.2501  | 0.1413  | 0.2290


<Figure size 640x480 with 0 Axes>